In [1]:
import numpy as np
import pandas as pd

# ---- same function as in notebook ----
def compute_difference_norms(df1, df2):
    d = pd.DataFrame()
    d['diff_X'] = df1['X'] - df2['X']
    d['diff_Y'] = df1['Y'] - df2['Y']
    d['diff_Z'] = df1['Z'] - df2['Z']
    d['error_norm'] = np.sqrt(
        d['diff_X']**2 + d['diff_Y']**2 + d['diff_Z']**2
    )
    return d['error_norm']


# ---- same ground truth loading as notebook ----
def load_ground_truth():
    return pd.read_csv(
        "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"
    )[["X", "Y", "Z"]]


# ---- filename patterns copied from your notebook ----
patterns = {
    ("gps","absolute"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","relative"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","log_d"):      "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("nns","absolute"):   "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","relative"):   "relative_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","log_d"):      "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("KANs","absolute"):  "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","relative"):  "KAN_predictions_relative_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","log_d"):     "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv"
}


# ============================================================
# FUNCTION TO COMPUTE STD OF P50 ACROSS 10 SEEDS
# ============================================================

def compute_std_of_p50(n, model_type, representation):
    df_gnd = load_ground_truth()
    
    base_dir = f"./predictions/{model_type}_{representation}/"
    pattern = patterns[(model_type, representation)]
    
    p50_values = []
    
    for seed in range(10):
        path = base_dir + pattern.format(n=n, seed=seed)
        
        df_pred = pd.read_csv(path)
        errors = compute_difference_norms(df_gnd, df_pred)
        
        p50 = errors.quantile(0.50)   # median error
        p50_values.append(p50)
    
    p50_values = np.array(p50_values)
    
    std_p50 = np.std(p50_values)
    
    return p50_values, std_p50


# ============================================================
# EXAMPLE: n = 400, Absolute NN
# ============================================================

n = 400
model_type = "nns"
representation = "absolute"

p50_values, std_p50 = compute_std_of_p50(n, model_type, representation)

print(f"P50 values across 10 seeds:\n{p50_values}")
print(f"\nStandard deviation of P50 (n={n}, {model_type}_{representation}): {std_p50:.6f} meters")

P50 values across 10 seeds:
[0.26552461 0.2955282  0.23552583 0.27579663 0.27555262 0.23978397
 0.25950314 0.25805951 0.254901   0.27086908]

Standard deviation of P50 (n=400, nns_absolute): 0.016863 meters


In [2]:
import numpy as np
import pandas as pd
from itertools import product


# --------------------------------------------------
# Same functions from before
# --------------------------------------------------

def compute_difference_norms(df1, df2):
    d = pd.DataFrame()
    d['diff_X'] = df1['X'] - df2['X']
    d['diff_Y'] = df1['Y'] - df2['Y']
    d['diff_Z'] = df1['Z'] - df2['Z']
    d['error_norm'] = np.sqrt(
        d['diff_X']**2 + d['diff_Y']**2 + d['diff_Z']**2
    )
    return d['error_norm']


def load_ground_truth():
    return pd.read_csv(
        "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"
    )[["X", "Y", "Z"]]


patterns = {
    ("gps","absolute"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","relative"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","log_d"):      "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("nns","absolute"):   "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","relative"):   "relative_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","log_d"):      "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("KANs","absolute"):  "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","relative"):  "KAN_predictions_relative_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","log_d"):     "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv"
}


def compute_std_of_p50(n, model_type, representation):
    df_gnd = load_ground_truth()
    
    base_dir = f"./predictions/{model_type}_{representation}/"
    pattern = patterns[(model_type, representation)]
    
    p50_values = []
    
    for seed in range(10):
        path = base_dir + pattern.format(n=n, seed=seed)
        df_pred = pd.read_csv(path)
        errors = compute_difference_norms(df_gnd, df_pred)
        p50_values.append(errors.quantile(0.50))
    
    return np.std(p50_values)


# --------------------------------------------------
# MAIN LOOP
# --------------------------------------------------

model_types = ["gps", "nns", "KANs"]
representations = ["absolute", "relative", "log_d"]
training_sizes = list(range(50, 1501, 50))   # 50 → 1500


results = []

for model, rep, n in product(model_types, representations, training_sizes):
    
    try:
        std_p50 = compute_std_of_p50(n, model, rep)
        
        results.append({
            "model": model,
            "representation": rep,
            "n_train": n,
            "std_p50_meters": std_p50
        })
        
        print(f"Done: {model} | {rep} | n={n}")
        
    except Exception as e:
        print(f"Skipped: {model} | {rep} | n={n}  -> {e}")


df_results = pd.DataFrame(results)

df_results.head()

Done: gps | absolute | n=50
Done: gps | absolute | n=100
Done: gps | absolute | n=150
Done: gps | absolute | n=200
Done: gps | absolute | n=250
Done: gps | absolute | n=300
Done: gps | absolute | n=350
Done: gps | absolute | n=400
Done: gps | absolute | n=450
Done: gps | absolute | n=500
Done: gps | absolute | n=550
Done: gps | absolute | n=600
Done: gps | absolute | n=650
Done: gps | absolute | n=700
Done: gps | absolute | n=750
Done: gps | absolute | n=800
Done: gps | absolute | n=850
Done: gps | absolute | n=900
Done: gps | absolute | n=950
Done: gps | absolute | n=1000
Done: gps | absolute | n=1050
Done: gps | absolute | n=1100
Done: gps | absolute | n=1150
Done: gps | absolute | n=1200
Done: gps | absolute | n=1250
Done: gps | absolute | n=1300
Done: gps | absolute | n=1350
Done: gps | absolute | n=1400
Done: gps | absolute | n=1450
Done: gps | absolute | n=1500
Done: gps | relative | n=50
Done: gps | relative | n=100
Done: gps | relative | n=150
Done: gps | relative | n=200
Done:

,model,representation,n_train,std_p50_meters
0,gps,absolute,50,0.061927
1,gps,absolute,100,0.039053
2,gps,absolute,150,0.010306
3,gps,absolute,200,0.007316
4,gps,absolute,250,0.007637


In [3]:
import numpy as np
import pandas as pd
from itertools import product


# --------------------------------------------------
# SAME FUNCTIONS AS BEFORE
# --------------------------------------------------

def compute_difference_norms(df1, df2):
    d = pd.DataFrame()
    d['diff_X'] = df1['X'] - df2['X']
    d['diff_Y'] = df1['Y'] - df2['Y']
    d['diff_Z'] = df1['Z'] - df2['Z']
    d['error_norm'] = np.sqrt(
        d['diff_X']**2 + d['diff_Y']**2 + d['diff_Z']**2
    )
    return d['error_norm']


def load_ground_truth():
    return pd.read_csv(
        "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"
    )[["X", "Y", "Z"]]


patterns = {
    ("gps","absolute"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","relative"):   "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps","log_d"):      "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("nns","absolute"):   "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","relative"):   "relative_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns","log_d"):      "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("KANs","absolute"):  "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","relative"):  "KAN_predictions_relative_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs","log_d"):     "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv"
}


# --------------------------------------------------
# PARAMETERS
# --------------------------------------------------

model_types = ["gps", "nns", "KANs"]
representations = ["absolute", "relative", "log_d"]
training_sizes = [50, 100, 200, 400]


# --------------------------------------------------
# MAIN COMPUTATION
# --------------------------------------------------

df_gnd = load_ground_truth()
results = []

for model, rep, n in product(model_types, representations, training_sizes):
    
    base_dir = f"./predictions/{model}_{rep}/"
    pattern = patterns[(model, rep)]
    
    p50_list = []
    p95_list = []
    mean_list = []
    
    for seed in range(10):
        path = base_dir + pattern.format(n=n, seed=seed)
        
        df_pred = pd.read_csv(path)
        errors = compute_difference_norms(df_gnd, df_pred)
        
        p50_list.append(errors.quantile(0.50))
        p95_list.append(errors.quantile(0.95))
        mean_list.append(errors.mean())
    
    # Convert to numpy
    p50_arr = np.array(p50_list)
    p95_arr = np.array(p95_list)
    mean_arr = np.array(mean_list)
    
    results.append({
        "model": model,
        "representation": rep,
        "n_train": n,
        
        "avg_p50": p50_arr.mean(),
        "std_p50": p50_arr.std(),
        
        "avg_p95": p95_arr.mean(),
        "std_p95": p95_arr.std(),
        
        "avg_mean_error": mean_arr.mean(),
        "std_mean_error": mean_arr.std()
    })
    
    print(f"Done: {model} | {rep} | n={n}")


df_results = pd.DataFrame(results)

df_results

Done: gps | absolute | n=50
Done: gps | absolute | n=100
Done: gps | absolute | n=200
Done: gps | absolute | n=400
Done: gps | relative | n=50
Done: gps | relative | n=100
Done: gps | relative | n=200
Done: gps | relative | n=400
Done: gps | log_d | n=50
Done: gps | log_d | n=100
Done: gps | log_d | n=200
Done: gps | log_d | n=400
Done: nns | absolute | n=50
Done: nns | absolute | n=100
Done: nns | absolute | n=200
Done: nns | absolute | n=400
Done: nns | relative | n=50
Done: nns | relative | n=100
Done: nns | relative | n=200
Done: nns | relative | n=400
Done: nns | log_d | n=50
Done: nns | log_d | n=100
Done: nns | log_d | n=200
Done: nns | log_d | n=400
Done: KANs | absolute | n=50
Done: KANs | absolute | n=100
Done: KANs | absolute | n=200
Done: KANs | absolute | n=400
Done: KANs | relative | n=50
Done: KANs | relative | n=100
Done: KANs | relative | n=200
Done: KANs | relative | n=400
Done: KANs | log_d | n=50
Done: KANs | log_d | n=100
Done: KANs | log_d | n=200
Done: KANs | log

,model,representation,n_train,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,gps,absolute,50,0.891132,0.061927,2.974824,0.149444,1.143516,0.050567
1,gps,absolute,100,0.392031,0.039053,2.128626,0.117952,0.654947,0.035359
2,gps,absolute,200,0.195652,0.007316,1.410302,0.123709,0.380165,0.022632
3,gps,absolute,400,0.091892,0.009987,0.993811,0.101487,0.240208,0.013796
4,gps,relative,50,0.309451,0.041879,1.191913,0.263918,0.421292,0.076365
5,gps,relative,100,0.147276,0.007352,0.748877,0.070706,0.227588,0.013517
6,gps,relative,200,0.064086,0.004110,0.529126,0.052496,0.134927,0.008442
7,gps,relative,400,0.030301,0.002336,0.312502,0.046044,0.081748,0.007298
8,gps,log_d,50,0.044805,0.004589,0.503669,0.071479,0.125422,0.013280
9,gps,log_d,100,0.016347,0.001445,0.303517,0.052301,0.064103,0.007744


In [6]:
import pandas as pd

# --------------------------------------------------
# Define desired order
# --------------------------------------------------

representation_order = ["absolute", "relative", "log_d"]
model_order = ["gps", "nns", "KANs"]

# Make categorical ordering
df_results["representation"] = pd.Categorical(
    df_results["representation"],
    categories=representation_order,
    ordered=True
)

df_results["model"] = pd.Categorical(
    df_results["model"],
    categories=model_order,
    ordered=True
)


# --------------------------------------------------
# Function to create ordered table for a given n
# --------------------------------------------------

def create_table_for_n(n_value):
    df_n = df_results[df_results["n_train"] == n_value].copy()
    
    df_n = df_n.sort_values(["representation", "model"])
    
    # Drop n_train column since all rows share same n
    df_n = df_n.drop(columns=["n_train"])
    
    return df_n.reset_index(drop=True)


# --------------------------------------------------
# Create the four tables
# --------------------------------------------------

table_50  = create_table_for_n(50)
table_100 = create_table_for_n(100)
table_200 = create_table_for_n(200)
table_400 = create_table_for_n(400)

print("===== n = 50 =====")
display(table_50)

print("===== n = 100 =====")
display(table_100)

print("===== n = 200 =====")
display(table_200)

print("===== n = 400 =====")
display(table_400)

===== n = 50 =====


,model,representation,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,gps,absolute,0.891132,0.061927,2.974824,0.149444,1.143516,0.050567
1,nns,absolute,0.701375,0.111330,2.704535,0.366866,0.977924,0.121097
2,KANs,absolute,1.435356,0.344607,9.316704,2.997259,2.692359,0.579294
3,gps,relative,0.309451,0.041879,1.191913,0.263918,0.421292,0.076365
4,nns,relative,0.551269,0.042712,1.786565,0.171632,0.704489,0.058971
5,KANs,relative,1.517035,0.518879,6.628756,1.343327,2.244212,0.552606
6,gps,log_d,0.044805,0.004589,0.503669,0.071479,0.125422,0.013280
7,nns,log_d,0.239061,0.030433,0.718609,0.068630,0.298787,0.028076
8,KANs,log_d,0.784109,0.586478,3.372294,2.524674,1.231044,0.984354


===== n = 100 =====


,model,representation,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,gps,absolute,0.392031,0.039053,2.128626,0.117952,0.654947,0.035359
1,nns,absolute,0.467600,0.051724,1.711700,0.179563,0.642086,0.070010
2,KANs,absolute,1.166409,0.090600,4.930916,0.871303,2.004792,0.469647
3,gps,relative,0.147276,0.007352,0.748877,0.070706,0.227588,0.013517
4,nns,relative,0.335981,0.025512,1.137194,0.115248,0.438533,0.031658
5,KANs,relative,0.786898,0.243895,4.582159,1.317879,1.329855,0.376041
6,gps,log_d,0.016347,0.001445,0.303517,0.052301,0.064103,0.007744
7,nns,log_d,0.131742,0.003535,0.465783,0.019803,0.179362,0.003318
8,KANs,log_d,0.503671,0.490944,2.429807,2.141577,0.857497,0.879385


===== n = 200 =====


,model,representation,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,gps,absolute,0.195652,0.007316,1.410302,0.123709,0.380165,0.022632
1,nns,absolute,0.329463,0.018878,1.081786,0.085670,0.425792,0.021219
2,KANs,absolute,0.398504,0.102389,1.646305,0.438562,0.611104,0.121709
3,gps,relative,0.064086,0.004110,0.529126,0.052496,0.134927,0.008442
4,nns,relative,0.210765,0.013425,0.620822,0.052847,0.259694,0.017173
5,KANs,relative,0.439684,0.256038,2.464950,0.522215,0.738355,0.248767
6,gps,log_d,0.007339,0.000178,0.139439,0.014183,0.032400,0.002979
7,nns,log_d,0.095438,0.005206,0.279732,0.022714,0.121168,0.007391
8,KANs,log_d,0.141649,0.116681,0.969972,0.893045,0.281406,0.221687


===== n = 400 =====


,model,representation,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,gps,absolute,0.091892,0.009987,0.993811,0.101487,0.240208,0.013796
1,nns,absolute,0.263104,0.016863,0.726339,0.054458,0.316621,0.018577
2,KANs,absolute,0.270502,0.053669,1.063810,0.263836,0.423986,0.115225
3,gps,relative,0.030301,0.002336,0.312502,0.046044,0.081748,0.007298
4,nns,relative,0.163506,0.007177,0.400094,0.018335,0.189526,0.006737
5,KANs,relative,0.132378,0.037320,0.851734,0.376290,0.254984,0.087705
6,gps,log_d,0.004664,0.000204,0.060101,0.004377,0.016693,0.001377
7,nns,log_d,0.080341,0.009876,0.223480,0.021626,0.098335,0.010589
8,KANs,log_d,0.066942,0.004416,0.321204,0.025282,0.113874,0.008652


In [5]:
import numpy as np
import pandas as pd


# --------------------------------------------------
# SAME ERROR FUNCTION
# --------------------------------------------------

def compute_difference_norms(df1, df2):
    d = pd.DataFrame()
    d['diff_X'] = df1['X'] - df2['X']
    d['diff_Y'] = df1['Y'] - df2['Y']
    d['diff_Z'] = df1['Z'] - df2['Z']
    d['error_norm'] = np.sqrt(
        d['diff_X']**2 + d['diff_Y']**2 + d['diff_Z']**2
    )
    return d['error_norm']


# --------------------------------------------------
# LOAD CLEAN GROUND TRUTH (always noise=0.0)
# --------------------------------------------------

df_gnd = pd.read_csv(
    "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"
)[["X", "Y", "Z"]]


# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

n = 400
model = "gps"
representation = "log_d"

noise_levels = {
    "inf_SNR": 0.0,
    "60_dB": 6e-06,
    "50_dB": 1.9e-05,
    "40_dB": 6e-05,
    "30_dB": 1.9e-04
}


# --------------------------------------------------
# MAIN LOOP
# --------------------------------------------------

results = []

for snr_label, noise in noise_levels.items():
    
    p50_list = []
    p95_list = []
    mean_list = []
    
    base_dir = f"./predictions/{model}_{representation}/"
    
    for seed in range(1):
        
        filename = f"rss_n={n}_noise={noise}_seed={seed}.csv"
        path = base_dir + filename
        
        df_pred = pd.read_csv(path)
        errors = compute_difference_norms(df_gnd, df_pred)
        
        p50_list.append(errors.quantile(0.50))
        p95_list.append(errors.quantile(0.95))
        mean_list.append(errors.mean())
    
    p50_arr = np.array(p50_list)
    p95_arr = np.array(p95_list)
    mean_arr = np.array(mean_list)
    
    results.append({
        "SNR_level": snr_label,
        "noise_sigma": noise,
        
        "avg_p50": p50_arr.mean(),
        "std_p50": p50_arr.std(),
        
        "avg_p95": p95_arr.mean(),
        "std_p95": p95_arr.std(),
        
        "avg_mean_error": mean_arr.mean(),
        "std_mean_error": mean_arr.std()
    })
    
    print(f"Done: {snr_label}")


df_noise_results = pd.DataFrame(results)

df_noise_results

Done: inf_SNR
Done: 60_dB
Done: 50_dB
Done: 40_dB
Done: 30_dB


,SNR_level,noise_sigma,avg_p50,std_p50,avg_p95,std_p95,avg_mean_error,std_mean_error
0,inf_SNR,0.000000,0.004893,0.0,0.060875,0.0,0.019685,0.0
1,60_dB,0.000006,0.007382,0.0,0.066706,0.0,0.022914,0.0
2,50_dB,0.000019,0.015265,0.0,0.106085,0.0,0.036762,0.0
3,40_dB,0.000060,0.042482,0.0,0.269224,0.0,0.085892,0.0
4,30_dB,0.000190,0.130598,0.0,0.767024,0.0,0.233366,0.0
